# <span style="color:#1B4F72;">RNN Project Pipeline</span>

**Business Understanding**  
↓  
**Data Collection**  
↓  
**Dataset Understanding**  
> Check **sequence length, sampling rate, and ordering**

↓  
**Data Cleaning**  
↓  
**Handle Missing Values**  
> Time Series → **Interpolation / Forward-fill**  
> **NOT random imputation — order matters!**

↓  
**Handle Outliers** *(if required)*  
↓  
**Exploratory Data Analysis (EDA)**  
> Time Series → Plot sequences over time, autocorrelation, trend/seasonality  
> Text → Token/word frequency

↓  
**Feature Engineering** *(if required)*  
> Time Series → Lag features, rolling statistics  
> Text → Tokenization, vocabulary building

↓  
**Define Features (X) and Target (y)**  
↓  
**Create Sequences (Windowing)**  
> Slice data into **fixed-length sequences**  
>
> Example:  
> `Past 30 days → Predict Day 31`  
> `Sentence → Pad/Truncate → max_len`

↓  
**Train-Validation-Test Split**  
> Time Series → **Chronological split**  
> No random shuffling → prevents **future-to-past data leakage**  
> Text → Can still shuffle

↓  
**Encoding**  
> Time Series → Label Encoding  
> Text → Tokenization + Vocabulary/Embedding Index

↓  
**Feature Scaling**  
> Time Series → MinMax / Standard Scaling  
> Text → Usually skipped; embeddings handle representation

↓  
**Convert to Tensors**

```text
Traditional:
(batch, features)

RNN:
(batch, seq_len, features)
```

↓  
**Create Dataset & DataLoader**  
> Variable-length sequences may require **padding / collate_fn**

↓  
**Design RNN Architecture**  
> Choose **RNN / LSTM / GRU**  
> Decide: `hidden_size`, `num_layers`, `bidirectional`

↓  
**Train the Model**  
> Watch for **vanishing/exploding gradients**  
> Consider **gradient clipping**

↓  
**Evaluate the Model**  
> Time Series → **RMSE / MAE**  
> Text → **Accuracy / F1 / Perplexity**

↓  
**Save Model & Artifacts**  
> Save model weights along with required artifacts such as **tokenizer/vocab or scaler**

In [1]:
import pandas as pd
import numpy as np 

In [2]:
df = pd.read_csv(r"C:\Users\rohit\Downloads\AI and Data Science\𝗗𝗲𝗲𝗽 𝗟𝗲𝗮𝗿𝗻𝗶𝗻𝗴 𝗣𝗿𝗼𝗷𝗲𝗰𝘁𝘀\RNN Projects\New folder\data\1_Daily_minimum_temps.csv")

df.head()

df.drop('Date', axis=1, inplace=True)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3650 entries, 0 to 3649
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Temp    3650 non-null   str  
dtypes: str(1)
memory usage: 40.5 KB


In [4]:
df.drop(df[df['Temp'] == '?0.2'].index, inplace=True)

In [5]:
#Dropping any row contain a question mark

df = df[~df.astype(str).apply(lambda row: row.str.contains(r'\?').any(), axis=1)]

In [6]:
df['Temp'] = df['Temp'].astype(float)

In [7]:
df.describe()

,Temp
count,3647.000000
mean,11.186647
std,4.061671
min,0.000000
25%,8.300000
50%,11.000000
75%,14.000000
max,26.300000


In [8]:
df.isnull().sum()

Temp    0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(3420)

In [10]:
df

,Temp
0,20.7
1,17.9
2,18.8
3,14.6
4,15.8
...,...
3645,14.0
3646,13.6
3647,13.5
3648,15.7


In [11]:
#preparing temp data 

temperatures = df['Temp'].values

print(temperatures[:10])
print(type(temperatures))
print(temperatures.shape)

[20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]
<class 'numpy.ndarray'>
(3647,)


In [12]:
#train - test - validation 

n = len(temperatures)

train_size = int(n * 0.70)
val_size = int(n * 0.15)

train_data = temperatures[:train_size]

val_data = temperatures[
    train_size:train_size + val_size
]

test_data = temperatures[
    train_size + val_size:
]

print("Total :", len(temperatures))
print("Train :", len(train_data))
print("Val   :", len(val_data))
print("Test  :", len(test_data))

Total : 3647
Train : 2552
Val   : 547
Test  : 548


In [16]:
n

3647

In [13]:
#scaling data

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(
    train_data.reshape(-1, 1)
)

val_scaled = scaler.transform(
    val_data.reshape(-1, 1)
)

test_scaled = scaler.transform(
    test_data.reshape(-1, 1)
)

In [14]:
def create_sequences(data, sequence_length):

    X = []
    y = []

    for i in range(len(data) - sequence_length):#Data ke andar window ko move karne ke liye loop chalao, jitni baar complete input sequence + target ban sakta hai.

        X.append(
            data[i:i + sequence_length]         #Current i se start karke sequence_length jitni values uthao aur X mein store karo.
        )

        y.append(
            data[i + sequence_length]
        )

    return np.array(X), np.array(y)

In [15]:
X_train, y_train = create_sequences(
    train_scaled,
    sequence_length
)

X_val, y_val = create_sequences(
    val_scaled,
    sequence_length
)

X_test, y_test = create_sequences(
    test_scaled,
    sequence_length
)

NameError: name 'sequence_length' is not defined